# Level 1 experiments - both trackers, both scenes, 10s clips

Fills the real gap left after `kaggle/tracked_demo/`: `intersection` has no processed
output at all yet, and neither source has a same-run ByteTrack-vs-BoT-SORT comparison on
identical input. Runs both trackers on both sources' 10s clips from the shared
`flytbase-hackathon-clips-compressed` dataset, single model (`visdrone-yolov11s` - already
the primary demo model; detector comparison isn't the point of this run, tracker comparison
is).

**Both trackers now draw with the same deterministic per-track color** (golden-ratio hue
spacing - the exact function already used by `pipeline/trajectory_trails.py` locally), not
ByteTrack-via-`supervision`-annotator vs BotSort-hardcoded-green as before - needed for a
fair visual comparison and to match the trails video's coloring.

Everything else - trails video, colour extraction, kinematics - is local post-processing on
the pulled `detections.csv` (see `pipeline/trajectory_trails.py`, `pipeline/colour.py`,
`pipeline/kinematics.py`, all already built + self-tested), against the exact same clip
file this notebook reads (`videos/clips/compressed/<source>_10s.MP4` locally - byte-identical
to this dataset's copy, confirmed by matching file size). Keeping this notebook GPU-work-only
keeps the Kaggle surface area - and iteration risk - as small as possible.

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}:", torch.cuda.get_device_name(i))

# Same numpy/scipy pin-then-install pattern as kaggle/tracked_demo/tracked_demo.ipynb -
# boxmot pulling in a newer numpy than scipy's compiled internals expect broke
# `import supervision` there (a real failure, not a guess); pinning before any install
# avoids repeating it.
import numpy
PINS = f'"numpy=={numpy.__version__}"'
try:
    import scipy
    PINS += f' "scipy=={scipy.__version__}"'
except ImportError:
    pass
print("Pinning for all subsequent installs:", PINS)

In [ ]:
!pip install -q ultralytics sahi huggingface_hub supervision {PINS}
# --no-deps keeps boxmot from bumping numpy/scipy off the pin above; its own deps are
# installed separately below, still numpy-pinned. Same version (22.0.0) and reasoning as
# kaggle/tracked_demo/tracked_demo.ipynb.
!pip install -q --no-deps "boxmot==22.0.0"
!pip install -q filterpy lapx scikit-learn timm click rich {PINS}

In [ ]:
import glob

SOURCES = ["multiroad", "intersection"]
TRACKERS = ["bytetrack", "botsort"]
DURATION = "10s"
MODEL_NAME = "visdrone-yolov11s"
MODEL_REPO = "dronefreak/visdrone-yolov11s"

print("Contents of /kaggle/input:")
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(" ", os.path.join(root, f))

video_paths = {}
for source in SOURCES:
    found = glob.glob(f"/kaggle/input/**/{source}_{DURATION}.MP4", recursive=True)
    assert found, f"No {source}_{DURATION}.MP4 found - see directory listing above"
    video_paths[source] = found[0]
    print(source, "->", found[0])

## Class mapping + shared per-track color

`track_color` is the exact function from `pipeline/trajectory_trails.py`, duplicated here
(this notebook runs standalone on Kaggle, not importing local repo files - same convention
as `tracked_demo.ipynb`'s class-mapping copy) so the boxes drawn here and the trails drawn
locally afterward use identical colors per track ID.

In [ ]:
import colorsys

VISDRONE_NAMES = {0: "pedestrian", 1: "people", 2: "bicycle", 3: "car", 4: "van",
                   5: "truck", 6: "tricycle", 7: "awning-tricycle", 8: "bus", 9: "motor", 10: "others"}
VISDRONE_TO_TAXONOMY = {
    "car": "car", "van": "LGV_candidate", "truck": "HGV_candidate", "bus": "bus",
    "motor": "motorcycle", "pedestrian": "pedestrian", "people": "pedestrian",
    "bicycle": "other", "tricycle": "other", "awning-tricycle": "other", "others": "other",
}


def track_color(track_id: int) -> tuple[int, int, int]:
    """Deterministic, visually distinct BGR color per track ID (golden-ratio hue spacing) -
    must stay byte-for-byte identical to pipeline/trajectory_trails.py's version."""
    hue = (track_id * 0.618033988749895) % 1.0
    r, g, b = colorsys.hsv_to_rgb(hue, 0.85, 1.0)
    return int(b * 255), int(g * 255), int(r * 255)

## Run: SAHI-tiled detection -> tracker -> colourful annotated video + CSV, per (source, tracker)

Both trackers now share one drawing path (manual cv2, `track_color(tid)`) instead of
ByteTrack going through `supervision`'s annotators while BotSort drew flat green - that
asymmetry made the two trackers' output videos not directly comparable.

In [ ]:
import csv
import json
import time

import cv2
import numpy as np
from huggingface_hub import hf_hub_download
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

weights_path = hf_hub_download(repo_id=MODEL_REPO, filename="best.pt")
# Same threshold as tracked_demo.ipynb - raised from the benchmark's 0.15 after that caused
# severe track-ID churn from tile-boundary-flicker detections.
CONFIDENCE_THRESHOLD = 0.35
ROOT_OUTPUT_DIR = "/kaggle/working/level1_experiments"


def make_tracker(tracker_type):
    if tracker_type == "bytetrack":
        import supervision as sv
        return sv.ByteTrack()
    if tracker_type == "botsort":
        import gdown
        import boxmot
        from boxmot.trackers.bbox.botsort import BotSort
        from boxmot.reid.core.reid import ReID

        # Pre-fetch the ReID weight ourselves - boxmot's own downloader calls gdown with a
        # `fuzzy=` kwarg our installed gdown doesn't have (confirmed on a real run, see
        # tracked_demo.ipynb / PLAN.md).
        weights_dir = os.path.join(os.path.dirname(boxmot.__file__), "..", "models")
        os.makedirs(weights_dir, exist_ok=True)
        reid_weights_path = os.path.join(weights_dir, "osnet_x0_25_msmt17.pt")
        if not os.path.exists(reid_weights_path):
            gdown.download(id="1sSwXSUlj4_tHZequ_iZ8w_Jh0VaRQMqF", output=reid_weights_path, quiet=False)

        device = "cuda:0" if torch.cuda.is_available() else "cpu"
        reid = ReID(path=reid_weights_path, device=device, half=torch.cuda.is_available())
        return BotSort(reid_model=reid.model, track_buffer=60, frame_rate=30, with_reid=True)
    raise ValueError(f"Unknown tracker_type: {tracker_type}")


for source in SOURCES:
    video_path = video_paths[source]
    for tracker_type in TRACKERS:
        output_name = MODEL_NAME if tracker_type == "bytetrack" else f"{MODEL_NAME}-botsort"
        print(f"\n=== {source} / {output_name} ===")
        model_dir = f"{ROOT_OUTPUT_DIR}/{source}/{output_name}"
        os.makedirs(model_dir, exist_ok=True)

        detection_model = AutoDetectionModel.from_pretrained(
            model_type="ultralytics", model_path=weights_path,
            confidence_threshold=CONFIDENCE_THRESHOLD, device="cuda:0" if torch.cuda.is_available() else "cpu",
        )
        tracker = make_tracker(tracker_type)

        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        n_frames_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        writer = cv2.VideoWriter(f"{model_dir}/video_raw.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

        rows = []
        t0 = time.time()
        frame_idx = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            result = get_sliced_prediction(
                frame, detection_model, slice_height=768, slice_width=768,
                overlap_height_ratio=0.2, overlap_width_ratio=0.2, verbose=0,
            )
            preds = result.object_prediction_list

            if tracker_type == "bytetrack":
                import supervision as sv
                if preds:
                    xyxy = np.array([p.bbox.to_xyxy() for p in preds], dtype=np.float32)
                    confidence = np.array([p.score.value for p in preds], dtype=np.float32)
                    class_id = np.array([p.category.id for p in preds], dtype=int)
                    detections = sv.Detections(xyxy=xyxy, confidence=confidence, class_id=class_id)
                else:
                    detections = sv.Detections.empty()
                tracked = tracker.update_with_detections(detections)
                track_rows = list(zip(tracked.xyxy, tracked.class_id, tracked.tracker_id, tracked.confidence))
            else:  # botsort - dets shape (N,6): x1,y1,x2,y2,conf,cls (confirmed on a local smoke test)
                if preds:
                    dets = np.array(
                        [[*p.bbox.to_xyxy(), p.score.value, p.category.id] for p in preds], dtype=np.float32
                    )
                else:
                    dets = np.empty((0, 6), dtype=np.float32)
                tracked_arr = np.asarray(tracker.update(dets, frame))
                # boxmot output columns: x1,y1,x2,y2,id,conf,cls,det_ind (confirmed locally)
                track_rows = [(row[:4], row[6], row[4], row[5]) for row in tracked_arr]

            annotated = frame.copy()
            for xyxy_box, cid, tid, conf in track_rows:
                native = VISDRONE_NAMES.get(int(cid), "others")
                taxonomy = VISDRONE_TO_TAXONOMY.get(native, "other")
                x1, y1, x2, y2 = [float(v) for v in xyxy_box]
                rows.append({
                    "frame": frame_idx, "time_s": round(frame_idx / fps, 2),
                    "track_id": int(tid), "native_class": native, "taxonomy_class": taxonomy,
                    "confidence": round(float(conf), 3),
                    "x1": round(x1, 1), "y1": round(y1, 1), "x2": round(x2, 1), "y2": round(y2, 1),
                })

                color = track_color(int(tid))
                p1, p2 = (int(x1), int(y1)), (int(x2), int(y2))
                cv2.rectangle(annotated, p1, p2, color, 2)
                label = f"#{int(tid)} {taxonomy}"
                cv2.putText(annotated, label, (p1[0], max(0, p1[1] - 5)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            writer.write(annotated)

            frame_idx += 1
            if frame_idx % 100 == 0:
                print(f"  frame {frame_idx}/{n_frames_total} ({time.time() - t0:.0f}s elapsed)")

        cap.release()
        writer.release()

        fieldnames = ["frame", "time_s", "track_id", "native_class", "taxonomy_class",
                      "confidence", "x1", "y1", "x2", "y2"]
        with open(f"{model_dir}/detections.csv", "w", newline="") as f:
            w_csv = csv.DictWriter(f, fieldnames=fieldnames)
            w_csv.writeheader()
            w_csv.writerows(rows)

        n_tracks = len(set(r["track_id"] for r in rows))
        meta = {
            "source_video": source,
            "duration_s": int(DURATION.rstrip("s")),
            "confidence_threshold": CONFIDENCE_THRESHOLD,
            "model": output_name,
            "weights_source": MODEL_REPO,
            "frames_processed": frame_idx,
            "fps": fps,
            "frame_width": w,
            "frame_height": h,
            "detection_rows": len(rows),
            "unique_tracks": n_tracks,
            "tracker": "supervision.ByteTrack" if tracker_type == "bytetrack" else "boxmot.BotSort (+ReID, track_buffer=60)",
            "sliced_inference": {"slice_size": 768, "overlap": 0.2, "tool": "SAHI"},
        }
        with open(f"{model_dir}/meta.json", "w") as f:
            json.dump(meta, f, indent=2)

        print(f"{source}/{output_name}: {frame_idx} frames, {len(rows)} detection rows, {n_tracks} unique tracks")

## Transcode to browser-compatible H.264

Same reason as `tracked_demo.ipynb` - OpenCV's `mp4v` output often won't play in a browser
`<video>` tag.

In [ ]:
for source in SOURCES:
    for tracker_type in TRACKERS:
        output_name = MODEL_NAME if tracker_type == "bytetrack" else f"{MODEL_NAME}-botsort"
        model_dir = f"{ROOT_OUTPUT_DIR}/{source}/{output_name}"
        src = f"{model_dir}/video_raw.mp4"
        dst = f"{model_dir}/video_h264.mp4"
        os.system(f'ffmpeg -y -i "{src}" -c:v libx264 -pix_fmt yuv420p -movflags +faststart "{dst}"')
        print(source, output_name, "->", dst, os.path.exists(dst))

## Next (local, no Kaggle)

Pull `/kaggle/working/level1_experiments` via `kaggle kernels output`. For each of the 4
`<source>/<model>` folders:

1. Drop `video_h264.mp4`, `detections.csv`, `meta.json` into
   `videos/samples/<source>/10s/<model>/`.
2. Run `pipeline/trajectory_trails.py` against the matching local
   `videos/clips/compressed/<source>_10s.MP4` (byte-identical to this dataset's copy) and
   this run's `detections.csv` to get `trails_h264.mp4` - no Kaggle needed, trails are pure
   post-processing.
3. Run `pipeline/colour.py` + `pipeline/kinematics.py` against the same clip + CSV to build
   the Level 2 `tracks_summary.csv` (per-track dominant colour, pixel-space speed/accel).